Which genres receive the highest ratings?
- book_id
- primary_genre
- average_rating
- ratings_count

In [46]:
import pandas as pd
books = pd.read_csv("../data/processed/sampled_books.csv")
book_genres = pd.read_json("../data/processed/sample_book_genres.csv", lines=True)

# remove all other columns except for book_id, average_rating, ratings_count
books = books[['book_id', 'average_rating', 'ratings_count']]

# shape before removing invalid data
before_books = books.shape
before_book_genres = book_genres.shape

# convert to numeric
for col in ["ratings_count", "average_rating"]:
    books[col] = pd.to_numeric(books[col], errors="coerce")

# remove books with zero ratings and books outside of ratings range
books = books[(books['ratings_count'] > 0) & (books["average_rating"].between(1.0, 5.0))]

# remove books where genres are empty 
book_genres = book_genres[book_genres["genres"].map(bool)]

# For each book find the highest number genre (most likely) and load into primary_genre column
book_genres["primary_genre"] = book_genres["genres"].apply(lambda g: max(g, key=g.get))

# merge books and genres datasets into a workable books_with_genres dataset 
books_with_genres = books.merge(book_genres[['book_id', 'primary_genre']], on='book_id')

# shape after removing invalid data
after_books = books.shape
after_book_genres = book_genres.shape

print(f'Books dataset - Removed {before_books[0] - after_books[0]} rows of invalid data. Shape is now {after_books}')
print(f'Books genres dataset - Removed {before_book_genres[0] - after_book_genres[0]} rows of invalid data. Shape is now {after_book_genres}')
print(f'Books with genres dataset - Shape of books_with_genres {books_with_genres.shape}')


Books dataset - Removed 649 rows of invalid data. Shape is now (117384, 3)
Books genres dataset - Removed 20489 rows of invalid data. Shape is now (97544, 3)
Books with genres dataset - Shape of books_with_genres (97248, 4)
